# DANNs on imbalanced data 

# 1. setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import lightning as L
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, RobustScaler  # ← Add RobustScaler here!
import os 
import matplotlib.pyplot as plt
import seaborn as sns
import psutil

print("="*60)
print("DANN EXPERIMENT: IMBALANCED DATASETS")
print("="*60)

torch.manual_seed(42)
np.random.seed(42)

# Memory check
print(f"Available memory: {psutil.virtual_memory().available / 1024**3:.1f} GB")

# 2. load imbalanced data

In [ ]:
c4_imbalanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_processed.csv')
ybt_imbalanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_processed.csv')

print(f"c4 shape: {c4_imbalanced.shape}")
print(f"ybt shape: {ybt_imbalanced.shape}")

# check class dist
print("c4 class dist:")
print(c4_imbalanced['autism_target'].value_counts())
print(f"c4 class balance: {c4_imbalanced['autism_target'].value_counts(normalize=True)}")

print("\nybt class dist:")
print(ybt_imbalanced['autism_target'].value_counts())
print(f"ybt class balance: {ybt_imbalanced['autism_target'].value_counts(normalize=True)}")

# memory usage after laoding 
print(f"memory usage: {psutil.virtual_memory().percent}%")


# 3. feature engineering (same as balanced)

In [ ]:
print("="*60)
print("FEATURE ENGINEERING FOR IMBALANCED DATA")
print("="*60)

# --- Robust Feature Engineering for Maximum Overlap ---

def create_aggregate_features(df, prefix, n_items):
    item_cols = [f"{prefix}_{i}" for i in range(1, n_items + 1) if f"{prefix}_{i}" in df.columns]
    if item_cols:
        df[f"{prefix}_total"] = df[item_cols].sum(axis=1)
    return df

for prefix, n_items in [('eq', 10), ('aq', 10), ('sqr', 10), ('spq', 10)]:
    c4_imbalanced = create_aggregate_features(c4_imbalanced, prefix, n_items)
    ybt_imbalanced = create_aggregate_features(ybt_imbalanced, prefix, n_items)

# D-score
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['d_score'] = df['eq_total'] - df['sqr_total'] 
    else:
        print(f"d_score not created for this dataframe")

# Age-EQ interaction 
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'age' in df.columns and 'eq_total' in df.columns:
        df['age_x_eq'] = df['age'] * df['eq_total']
    else:
        print(f"age_x_eq not created for this dataframe")

# Age-AQ interaction
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'age' in df.columns and 'aq_total' in df.columns:
        df['age_x_aq'] = df['age'] * df['aq_total']
    else:
        print(f"age_x_aq not created for this dataframe")

# Age-SQR interaction
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'age' in df.columns and 'sqr_total' in df.columns:  # ← CORRECT
        df['age_x_sqr'] = df['age'] * df['sqr_total']

# AQ-EQ interaction 
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'aq_total' in df.columns and 'eq_total' in df.columns:
        df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
    else:
        print(f"aq_eq_interaction not created for this dataframe")

# EQ/SQR ratio
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)
    else:
        print(f"eq_sqr_ratio not created for this dataframe")

# Log-transformed AQ total 
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'aq_total' in df.columns:
        df['log_aq_total'] = np.log1p(np.clip(df['aq_total'], a_min=0, a_max=None))

# Square root of age
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'age' in df.columns:
        df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))

# High AQ flag
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'aq_total' in df.columns:
        df['high_aq'] = (df['aq_total'] > 32).astype(int)

# Now recompute feature lists
exclude_cols = ['autism_target']  # Only exclude target

# Add these only if they exist
if 'userid' in c4_imbalanced.columns:
    exclude_cols.append('userid')
if 'date' in c4_imbalanced.columns:
    exclude_cols.append('date')
if 'timestamp' in c4_imbalanced.columns:
    exclude_cols.append('timestamp')

c4_features = [col for col in c4_imbalanced.columns if col not in exclude_cols]
ybt_features = [col for col in ybt_imbalanced.columns if col not in exclude_cols]

common_features = sorted(list(set(c4_features) & set(ybt_features)))
print(f"Common features after alignment: {len(common_features)}")
print("Sample common features:", common_features[:5])

# Print missing features 
missing_in_ybt = set(c4_features) - set(ybt_features)
missing_in_c4 = set(ybt_features) - set(c4_features)
print(f"Features missing in YBT: {len(missing_in_ybt)}")
print(f"Features in YBT but missing in C4: {len(missing_in_c4)}")

# Memory check
print(f"Memory usage: {psutil.virtual_memory().percent}%")

# 4. data preperation (imbalanced)

In [ ]:
print("DATA PREPERATION: IMBALANCED DATA")

# recreate data arrays with new features
x_c4_imb = c4_imbalanced[common_features].values
y_c4_imb = c4_imbalanced['autism_target'].values
x_ybt_imb = ybt_imbalanced[common_features].values
y_ybt_imb = ybt_imbalanced['autism_target'].values

print(f"original c4 shape:{x_c4_imb.shape}")
print(f"original ybt shape:{x_ybt_imb.shape}")

# standardization
scaler_imb = RobustScaler()
x_c4_imb_scaled = scaler_imb.fit_transform(x_c4_imb)
x_ybt_imb_scaled = scaler_imb.transform(x_ybt_imb)

# split c4 imbalanced data 
x_train_imb, x_val_imb, y_train_imb, y_val_imb = train_test_split(
    x_c4_imb_scaled, y_c4_imb, test_size=0.2, random_state=42, stratify=y_c4_imb
)

print(f"X_train_imb shape: {x_train_imb.shape}")
print(f"X_val_imb shape: {x_val_imb.shape}")
print(f"X_ybt_imb_scaled shape: {x_ybt_imb_scaled.shape}")

print(f"Train class distribution: {np.bincount(y_train_imb)}")
print(f"val class distribution: {np.bincount(y_val_imb)}")
print(f"YBT class distribution: {np.bincount(y_ybt_imb)}")

print("NaNs in x_train_imb:", np.isnan(x_train_imb).sum())
print("infs in x_train_imb:", np.isinf(x_train_imb).sum())

# memory usage 
print(f"memory usage: {psutil.virtual_memory().percent}%")

# 5. gradient reversal layer

In [ ]:
print("\n" + "="*60)
print("GRADIENT REVERSAL LAYER")
print("="*60)

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

# 6. DANN model 

In [ ]:
print("\n" + "="*60)
print("IMPROVED DOMAIN ADVERSARIAL CLASSIFIER (IMBALANCED)")
print("="*60)

class ImprovedDomainAdversarialClassifier(L.LightningModule):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.3, 
                 learning_rate=0.0005, alpha=1.0, domain_weight=0.05):
        super().__init__()
        self.save_hyperparameters()
        
        # Feature extractor
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = hidden_dim
        
        self.feature_extractor = nn.Sequential(*layers)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dims[-1], 1),
            nn.Sigmoid()
        )
        
        # Domain discriminator
        self.domain_discriminator = nn.Sequential(
            GradientReversalLayer(alpha),
            nn.Linear(hidden_dims[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
        self.class_criterion = nn.BCELoss()
        self.domain_criterion = nn.BCELoss()
        self.domain_weight = domain_weight

    def forward(self, x):
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        domain_output = self.domain_discriminator(features)
        return class_output, domain_output

    def training_step(self, batch, batch_idx):
        x, y, domain = batch
        
        class_output, domain_output = self(x)
        
        # Classification loss
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # Domain discrimination loss
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        
        # Total loss
        total_loss = class_loss + self.domain_weight * domain_loss
        
        # Metrics
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        
        self.log('train_loss', total_loss)
        self.log('train_class_loss', class_loss)
        self.log('train_domain_loss', domain_loss)
        self.log('train_f1', f1)
        
        return total_loss

    def validation_step(self, batch, batch_idx):
        x, y, domain = batch
        
        class_output, domain_output = self(x)
        
        # Classification loss
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # Domain discrimination loss
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        
        # Total loss
        total_loss = class_loss + self.domain_weight * domain_loss
        
        # Metrics
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        
        self.log('val_loss', total_loss)
        self.log('val_class_loss', class_loss)
        self.log('val_domain_loss', domain_loss)
        self.log('val_f1', f1)
        
        return total_loss

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.7, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_f1"
            }
        }

# 7. data module

In [ ]:
print("\n" + "="*60)
print("IMPROVED DOMAIN ADAPTATION DATA MODULE (IMBALANCED)")
print("="*60)

class ImprovedDomainAdaptationDataModule(L.LightningDataModule):
    def __init__(self, X_train, X_val, y_train, y_val, X_test, y_test, batch_size=32):
        super().__init__()
        self.X_train = X_train
        self.X_val = X_val
        self.y_train = y_train
        self.y_val = y_val
        self.X_test = X_test
        self.y_test = y_test
        self.batch_size = batch_size

    def setup(self, stage=None):
        # Create datasets
        self.train_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_train),
            torch.LongTensor(self.y_train),
            torch.zeros(len(self.X_train))  # Domain 0 for C4
        )
        
        self.val_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_val),
            torch.LongTensor(self.y_val),
            torch.zeros(len(self.X_val))  # Domain 0 for C4
        )
        
        self.test_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_test),
            torch.LongTensor(self.y_test),
            torch.ones(len(self.X_test))  # Domain 1 for YBT
        )

    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset, batch_size=self.batch_size, shuffle=True,
            num_workers=4, pin_memory=True
        )

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_dataset, batch_size=self.batch_size, shuffle=False,
            num_workers=4, pin_memory=True
        )

    def test_dataloader(self):
        return torch.utils.data.DataLoader(
            self.test_dataset, batch_size=self.batch_size, shuffle=False,
            num_workers=4, pin_memory=True
        )

# Create data module for imbalanced experiment
data_module_imb = ImprovedDomainAdaptationDataModule(
    x_train_imb, x_val_imb, y_train_imb, y_val_imb, x_ybt_imb_scaled, y_ybt_imb, batch_size=32
)

# 8. train imbalanced DANN

In [ ]:
print("\n" + "="*60)
print("TRAINING IMBALANCED DANN MODEL")
print("="*60)

# Initialize model for imbalanced data
model_imb = ImprovedDomainAdversarialClassifier(
    input_dim=len(common_features),
    hidden_dims=[128, 64, 32],
    dropout_rate=0.3,
    learning_rate=0.0005,
    alpha=1.0,
    domain_weight=0.05
)

# Trainer with imbalanced-appropriate settings
trainer_imb = L.Trainer(
    max_epochs=100,
    accelerator='auto',
    devices=1,
    callbacks=[
        L.pytorch.callbacks.EarlyStopping(monitor='val_f1', patience=15, mode='max', verbose=True),
        L.pytorch.callbacks.ModelCheckpoint(monitor='val_f1', mode='max', save_top_k=3, verbose=True),
        L.pytorch.callbacks.LearningRateMonitor(logging_interval='epoch')
    ],
    log_every_n_steps=25,
    enable_progress_bar=True,
    enable_model_summary=True,
    deterministic=True
)

# Train the model
print("Training imbalanced DANN model...")
trainer_imb.fit(model_imb, data_module_imb)

print("Imbalanced DANN training completed!")
print(f"Best validation F1: {trainer_imb.checkpoint_callback.best_model_score:.3f}")

# 9. test balanced DANN on imbalanced data 

In [ ]:
print("\n" + "="*60)
print("TESTING BALANCED DANN ON IMBALANCED DATA")
print("="*60)

# Load the balanced model (from your previous experiment)
# Load the balanced model from the previous experiment
balanced_model = ImprovedDomainAdversarialClassifier.load_from_checkpoint(
    '/Users/eb2007/playground/bullpy/c4_play2/models/dann_improved.pth'
)
balanced_model.eval()

# Test on imbalanced YBT data
ybt_predictions_balanced = []
ybt_probs_balanced = []
ybt_targets_balanced = []

x_ybt_tensor_imb = torch.FloatTensor(x_ybt_imb_scaled)
ybt_dataset_imb = torch.utils.data.TensorDataset(
    x_ybt_tensor_imb, torch.LongTensor(y_ybt_imb), torch.ones(len(x_ybt_imb_scaled))
)
ybt_dataloader_imb = torch.utils.data.DataLoader(ybt_dataset_imb, batch_size=32, shuffle=False)

device = next(balanced_model.parameters()).device

with torch.no_grad():
    for batch in ybt_dataloader_imb:
        x, y, domain = batch
        x = x.to(device)
        class_output, domain_output = balanced_model(x)
        ybt_probs_balanced.extend(class_output.squeeze().cpu().numpy())
        ybt_predictions_balanced.extend((class_output.squeeze() > 0.5).cpu().numpy())
        ybt_targets_balanced.extend(y.cpu().numpy())

ybt_probs_balanced = np.array(ybt_probs_balanced)
ybt_predictions_balanced = np.array(ybt_predictions_balanced)
ybt_targets_balanced = np.array(ybt_targets_balanced)

# Calculate metrics
balanced_on_imb_f1 = f1_score(ybt_targets_balanced, ybt_predictions_balanced, average='weighted')
balanced_on_imb_auc = roc_auc_score(ybt_targets_balanced, ybt_probs_balanced)

print(f"Balanced DANN on imbalanced YBT:")
print(f"F1 Score: {balanced_on_imb_f1:.3f}")
print(f"ROC-AUC: {balanced_on_imb_auc:.3f}")
print(f"Class distribution: {np.bincount(ybt_targets_balanced)}")

# 10. test imbalanced DANN on Imbalanced data 

In [ ]:
print("\n" + "="*60)
print("TESTING IMBALANCED DANN ON IMBALANCED DATA")
print("="*60)

# Load the best imbalanced model
imb_model = ImprovedDomainAdversarialClassifier.load_from_checkpoint(
    trainer_imb.checkpoint_callback.best_model_path
)
imb_model.eval()

# Test on imbalanced YBT data
ybt_predictions_imb = []
ybt_probs_imb = []
ybt_targets_imb = []

with torch.no_grad():
    for batch in ybt_dataloader_imb:
        x, y, domain = batch
        x = x.to(device)
        class_output, domain_output = imb_model(x)
        ybt_probs_imb.extend(class_output.squeeze().cpu().numpy())
        ybt_predictions_imb.extend((class_output.squeeze() > 0.5).cpu().numpy())
        ybt_targets_imb.extend(y.cpu().numpy())

ybt_probs_imb = np.array(ybt_probs_imb)
ybt_predictions_imb = np.array(ybt_predictions_imb)
ybt_targets_imb = np.array(ybt_targets_imb)

# Calculate metrics
imb_on_imb_f1 = f1_score(ybt_targets_imb, ybt_predictions_imb, average='weighted')
imb_on_imb_auc = roc_auc_score(ybt_targets_imb, ybt_probs_imb)

print(f"Imbalanced DANN on imbalanced YBT:")
print(f"F1 Score: {imb_on_imb_f1:.3f}")
print(f"ROC-AUC: {imb_on_imb_auc:.3f}")
print(f"Class distribution: {np.bincount(ybt_targets_imb)}")

# 11. comprehensive comparison 

In [ ]:
print("\n" + "="*60)
print("COMPREHENSIVE COMPARISON: BALANCED VS IMBALANCED")
print("="*60)

# Create comparison table
comparison_results = {
    'Experiment': [
        'Balanced DANN on Balanced Data',
        'Balanced DANN on Imbalanced Data', 
        'Imbalanced DANN on Imbalanced Data'
    ],
    'F1_Score': [
        0.729,  # Your previous balanced result
        balanced_on_imb_f1,
        imb_on_imb_f1
    ],
    'ROC_AUC': [
        0.787,  # Your previous balanced result
        balanced_on_imb_auc,
        imb_on_imb_auc
    ],
    'Training_Data': ['Balanced C4', 'Balanced C4', 'Imbalanced C4'],
    'Test_Data': ['Balanced YBT', 'Imbalanced YBT', 'Imbalanced YBT']
}

comparison_df = pd.DataFrame(comparison_results)
print("\nPerformance Comparison:")
print(comparison_df)

# Analysis
print(f"\nKey Insights:")
print(f"1. Balanced training → Balanced testing: F1={comparison_results['F1_Score'][0]:.3f}")
print(f"2. Balanced training → Imbalanced testing: F1={comparison_results['F1_Score'][1]:.3f}")
print(f"3. Imbalanced training → Imbalanced testing: F1={comparison_results['F1_Score'][2]:.3f}")

performance_drop = comparison_results['F1_Score'][0] - comparison_results['F1_Score'][1]
performance_gain = comparison_results['F1_Score'][2] - comparison_results['F1_Score'][1]

print(f"\nPerformance Analysis:")
print(f"- Drop from balanced to imbalanced testing: {performance_drop:.3f}")
print(f"- Gain from imbalanced training: {performance_gain:.3f}")

if performance_gain > 0:
    print(f"✅ Imbalanced training improves performance on real-world data")
else:
    print(f"⚠️ Imbalanced training does not improve performance")

# Save results
results_summary = {
    'balanced_on_balanced': {'f1': 0.729, 'auc': 0.787},
    'balanced_on_imbalanced': {'f1': balanced_on_imb_f1, 'auc': balanced_on_imb_auc},
    'imbalanced_on_imbalanced': {'f1': imb_on_imb_f1, 'auc': imb_on_imb_auc}
}

print(f"\nResults saved for publication!")